# DEKM 2021 paper-era: all three text datasets, seeds 42-44

This notebook runs one immutable paper-era DEKM contract on REUTERS, 20NEWS, and RCV1. It uses the released unit-L2 preprocessing, uniform initializer, `np.linalg.eig`, global-RNG K-Means, released 0.5% stop threshold, and the dataset-specific paper-era batch policy: REUTERS 10,000; 20NEWS and RCV1 256. Results are compared only with the DEKM row in Table III, using three-run mean and sample standard deviation.


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('GPUs:', gpus)
assert gpus, 'Enable a GPU accelerator in Kaggle notebook settings.'


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/kaggle/working/mvdec_fpmax')
subprocess.run(['rm', '-rf', str(REPO)], check=True)
subprocess.run([
    'git', 'clone', '--branch', 'dev', '--single-branch',
    'https://github.com/vdat-18/mvdec_fpmax.git', str(REPO),
], check=True)
EXPECTED_REVISION = '3f2f3ded53754b0b655e657e6a949c528fefda29'
subprocess.run(['git', 'checkout', '--detach', EXPECTED_REVISION], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO), 'gdown'], check=True)
actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert actual == EXPECTED_REVISION, actual
print('Repository revision:', actual)

subprocess.run([
    sys.executable, str(REPO / 'scripts/fetch_dekm_release.py'),
    '--datasets', 'REUTERS', 'RCV1',
], cwd=REPO, check=True)
from sklearn.datasets import fetch_20newsgroups, fetch_rcv1
fetch_20newsgroups(subset='all', download_if_missing=True)
fetch_rcv1(download_if_missing=True)
print('All three dataset sources are ready.')


In [ ]:
import json
import traceback

DATASETS = ('REUTERS', '20NEWS', 'RCV1')
SEEDS = (42, 43, 44)
PROTOCOL = 'dekm_2021_release_seeded_modern_runtime_v1'
OUTPUT = Path('/kaggle/working/dekm-2021-three-datasets-seeds42-44')
RUN_ROOT = OUTPUT / 'runs'
OUTPUT.mkdir(parents=True, exist_ok=True)
failures = []

for dataset in DATASETS:
    command = [
        sys.executable, '-m', 'representation_learning.DEKM_dense', dataset,
        '--protocol', 'release-2021', '--seed', str(SEEDS[0]),
        '--runs', str(len(SEEDS)), '--require-gpu',
        '--dataset-root', str(REPO / 'external_repos/DEKM/datasets'),
        '--output-dir', str(RUN_ROOT),
    ]
    try:
        subprocess.run(command, cwd=REPO, check=True)
    except Exception as error:
        failures.append({'dataset': dataset, 'seeds': list(SEEDS), 'error_type': type(error).__name__, 'error': str(error)})
        traceback.print_exc()
(OUTPUT / 'failures.json').write_text(json.dumps(failures, indent=2), encoding='utf-8')
print('Failures:', json.dumps(failures, indent=2))


In [ ]:
import hashlib
import pandas as pd
from pipeline.external_metrics import compute_external_metrics

PAPER = {'REUTERS': {'acc': 0.7628, 'nmi': 0.5906}, '20NEWS': {'acc': 0.4108, 'nmi': 0.4027}, 'RCV1': {'acc': 0.6715, 'nmi': 0.4618}}
EXPECTED_BATCH = {'REUTERS': 10000, '20NEWS': 256, 'RCV1': 256}
run_frames = []
manifest_paths = []

for dataset in DATASETS:
    protocol_root = RUN_ROOT / dataset.lower() / PROTOCOL
    config_roots = [path for path in protocol_root.iterdir() if path.is_dir()]
    assert len(config_roots) == 1, (dataset, config_roots)
    config_root = config_roots[0]
    run_frames.append(pd.read_csv(config_root / 'dekm_runs.csv'))
    for seed in SEEDS:
        manifest_path = config_root / f'seed_{seed}' / 'manifest.json'
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        assert manifest['status'] == 'complete'
        assert manifest['method'] == 'DEKM'
        assert manifest['protocol_id'] == PROTOCOL
        assert manifest['config']['refinement_batch_size'] == EXPECTED_BATCH[dataset]
        assert manifest['config']['assignment_change_tolerance'] == 0.005
        assert manifest['config']['input_normalization'] == 'release_l2_unit_norm'
        assert manifest['official_repository_paper_era_revision'] == '129b7d08b545f02d920ccff0948ec44b654b3158'
        assignment_path = manifest_path.parent / 'assignments.csv'
        assignment = pd.read_csv(assignment_path)
        metrics = compute_external_metrics(assignment['true_label'].to_numpy(), assignment['cluster'].to_numpy(), n_clusters=manifest['n_clusters'])
        assert abs(metrics.acc - manifest['metrics']['acc']) < 1e-12
        assert abs(metrics.nmi - manifest['metrics']['nmi']) < 1e-12
        for name, filename in manifest['paths'].items():
            path = manifest_path.parent / filename
            assert hashlib.sha256(path.read_bytes()).hexdigest() == manifest['output_sha256'][name]
        manifest_paths.append(manifest_path)

runs = pd.concat(run_frames, ignore_index=True).sort_values(['dataset', 'seed'])
assert len(runs) == 9
aggregate_rows = []
for dataset, group in runs.groupby('dataset', sort=True):
    reference = PAPER[dataset]
    aggregate_rows.append({'dataset': dataset, 'protocol_id': PROTOCOL, 'completed_runs': int(len(group)), 'acc_mean': float(group['acc'].mean()), 'acc_sample_std': float(group['acc'].std(ddof=1)), 'nmi_mean': float(group['nmi'].mean()), 'nmi_sample_std': float(group['nmi'].std(ddof=1)), 'paper_acc': reference['acc'], 'paper_nmi': reference['nmi'], 'acc_delta_pp': float(100 * (group['acc'].mean() - reference['acc'])), 'nmi_delta_pp': float(100 * (group['nmi'].mean() - reference['nmi'])), 'runtime_seconds_mean': float(group['fit_time_seconds'].mean()), 'source_sha256': group['source_sha256'].iloc[0], 'config_hash': group['config_hash'].iloc[0]})
aggregate = pd.DataFrame(aggregate_rows)
runs.to_csv(OUTPUT / 'dekm_runs.csv', index=False)
aggregate.to_csv(OUTPUT / 'dekm_aggregate.csv', index=False)
(OUTPUT / 'dekm_aggregate.json').write_text(aggregate.to_json(orient='records', indent=2), encoding='utf-8')
display(runs[['dataset', 'seed', 'acc', 'nmi', 'fit_time_seconds', 'stop_reason']])
display(aggregate)
assert not failures, failures


In [ ]:
import shutil

zip_path = Path(shutil.make_archive('/kaggle/working/dekm-2021-three-datasets-seeds42-44', 'zip', '/kaggle/working', 'dekm-2021-three-datasets-seeds42-44'))
digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()
sha_path = zip_path.with_suffix('.zip.sha256')
sha_path.write_text(f'{digest}  {zip_path.name}\n', encoding='ascii')
print('ZIP:', zip_path)
print('SHA-256:', digest)
print('SHA file:', sha_path)
